In [23]:
import pandas as pd
import os

DATA_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/filtered_dataset/data_partitions/no_valid/"

In [24]:
def data_cycle_oversampling(df: "pd.DataFrame", target: int = 100) -> "pd.DataFrame":
    """Return a copy of df padded with its own rows (in order, cyclically)
    until it reaches target rows.
    """
    df_out = df.copy()
    if df_out.shape[0] >= target:
        return df_out

    i = 0
    while df_out.shape[0] < target:
        df_out = pd.concat([df_out, df.iloc[[i]]], ignore_index=True)
        i = (i + 1) % len(df)
    return df_out


# Aca hay que hacer que este df de test reemplace la familia en train.

In [36]:
fams = ["23s", "5s", "RNaseP", "grp1", "srp", "tRNA", "telomerase", "tmRNA", "16s"]

df = pd.read_csv(DATA_PATH + "ArchiveII_hc_100.csv")
df

,fold,partition,id
0,16s,test,16s_A.fulgidus_domain2
1,16s,test,16s_A.fulgidus_domain3
2,16s,test,16s_A.fulgidus_domain4
3,16s,test,16s_A.pyrophilus_domain2
4,16s,test,16s_A.pyrophilus_domain4
...,...,...,...
9379,tmRNA,train,telomerase_AF221939.99-544
9380,tmRNA,train,telomerase_AF221940.103-499
9381,tmRNA,train,telomerase_AY058901.1-397
9382,tmRNA,train,telomerase_AY312571.605-1069


In [64]:
import pandas as pd


def data_partition_oversampling(df, target=100):
    # Extraer familia
    df["fam"] = df["id"].str.partition("_")[0]
    fams = df["fam"].unique()
    # para cada fold de entrenamiento (fold == familia en test)
    df_collector = []
    for fold in fams:
        df_train = df[(df["partition"] == "train") & (df["fold"] == fold)]
        df_test = df[(df["partition"] == "test") & (df["fold"] == fold)]
        # para cada familia posible reviso que todas tengan 100 elementos
        for f in fams:
            if f != fold:
                df_t_fam = df_train[df_train["fam"] == f]
                df_t_fam = data_cycle_oversampling(df_t_fam, target)
                df_collector.append(df_t_fam)
            # tambien colecto test para unirlo todo
            else:
                df_collector.append(df_test)
        df_final = pd.concat(df_collector, ignore_index=True)
    print(
        df_final.shape[0],
        df.shape[0],
    )
    return df_final[["fold", "fam", "id"]]

In [65]:
df = pd.read_csv(DATA_PATH + "ArchiveII_hc_100.csv")
df_ = data_partition_oversampling(df, 100)

11064 9384


In [66]:
pd.set_option("display.max_rows", 30)

df.groupby(["fold", "fam"])["id"].count().reset_index()

,fold,fam,id
0,16s,16s,66
1,16s,23s,15
2,16s,5s,100
3,16s,RNaseP,100
4,16s,grp1,74
...,...,...,...
76,tmRNA,grp1,74
77,tmRNA,srp,100
78,tmRNA,tRNA,100
79,tmRNA,telomerase,35


In [67]:
pd.set_option("display.max_rows", 30)

df_.groupby(["fold", "fam"])["id"].count().reset_index()

,fold,fam,id
0,16s,16s,66
1,16s,23s,100
2,16s,5s,100
3,16s,RNaseP,100
4,16s,grp1,100
...,...,...,...
76,tmRNA,grp1,100
77,tmRNA,srp,100
78,tmRNA,tRNA,100
79,tmRNA,telomerase,100


In [ ]:
pd.set_option("display.max_rows", None)
df_[(df_["fold"] == "16s") & (df_["fam"] == "23s")]

,fold,partition,id,fam
10198,16s,train,23s_B.subtilis_domain3,23s
10199,16s,train,23s_B.subtilis_domain4,23s
10200,16s,train,23s_B.subtilis_domain6,23s
10201,16s,train,23s_E.coli_domain3,23s
10202,16s,train,23s_E.coli_domain4,23s
10203,16s,train,23s_E.coli_domain6,23s
10204,16s,train,23s_H.pylori_domain3,23s
10205,16s,train,23s_H.pylori_domain4,23s
10206,16s,train,23s_H.pylori_domain6,23s
10207,16s,train,23s_S.aureus_domain3,23s


In [ ]:
SAVE = False
SAVE_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/filtered_dataset/data_partitions/over_sampled/"
methods = ["rnadist", "hc", "samples"]
thresholds = [100, 200, 400]

for method in methods:
    for thershold in thresholds:
        file = f"ArchiveII_{method}_{thershold}.csv"
        print(file)
        df = pd.read_csv(DATA_PATH + file)
        df = data_partition_oversampling(df, thershold)
        if SAVE:
            df.to_csv(SAVE_PATH + file)

ArchiveII_rnadist_100.csv
11064 9384
ArchiveII_rnadist_200.csv
18264 13384
ArchiveII_rnadist_400.csv
32664 21384
ArchiveII_hc_100.csv
11064 9384
ArchiveII_hc_200.csv
18264 13384
ArchiveII_hc_400.csv
32664 21384
ArchiveII_samples_100.csv
11064 9384
ArchiveII_samples_200.csv
18264 13384
ArchiveII_samples_400.csv
32664 21384


In [70]:
df.groupby(["fold", "fam"])["id"].count().reset_index()

,fold,fam,id
0,16s,16s,66
1,16s,23s,400
2,16s,5s,400
3,16s,RNaseP,400
4,16s,grp1,400
...,...,...,...
76,tmRNA,grp1,400
77,tmRNA,srp,400
78,tmRNA,tRNA,400
79,tmRNA,telomerase,400


In [69]:
display(df.head())

,fold,fam,id
0,16s,16s,16s_A.fulgidus_domain2
1,16s,16s,16s_A.fulgidus_domain3
2,16s,16s,16s_A.fulgidus_domain4
3,16s,16s,16s_A.pyrophilus_domain2
4,16s,16s,16s_A.pyrophilus_domain4
